# Week 1: House Price Prediction

本 Notebook 是暑期 AI 工程实践训练营第一周完整实验。

你将完成：

1. 读取或生成数据集。
2. 完成第一次 EDA。
3. 做基础特征工程。
4. 训练线性回归模型。
5. 评估模型效果。
6. 保存图表、评估指标和模型文件。

请从上到下逐个 cell 运行。

## 1. 导入依赖并创建目录

这一部分会创建工程输出目录：`data/`、`figures/`、`models/`、`results/`。

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT_DIR / "data"
FIGURE_DIR = ROOT_DIR / "figures"
MODEL_DIR = ROOT_DIR / "models"
RESULT_DIR = ROOT_DIR / "results"

for directory in [DATA_DIR, FIGURE_DIR, MODEL_DIR, RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("工程根目录:", ROOT_DIR)
print("数据目录:", DATA_DIR)
print("图表目录:", FIGURE_DIR)
print("模型目录:", MODEL_DIR)
print("结果目录:", RESULT_DIR)

## 2. 获取数据

优先下载 scikit-learn 内置的 California Housing Dataset。

如果下载失败，Notebook 会自动生成一个离线练习数据集，保证后续步骤可以继续运行。

In [ ]:
def make_offline_dataset(n=300, random_state=42):
    rng = np.random.default_rng(random_state)
    study_hours = rng.normal(5, 2, n).clip(0, 10)
    attendance_rate = rng.normal(0.85, 0.1, n).clip(0.4, 1.0)
    homework_score = rng.normal(80, 12, n).clip(30, 100)
    sleep_hours = rng.normal(7, 1, n).clip(4, 10)

    price = (
        8 * study_hours
        + 35 * attendance_rate
        + 0.45 * homework_score
        + 2 * sleep_hours
        + rng.normal(0, 5, n)
    ).clip(0, 100)

    return pd.DataFrame({
        "study_hours": study_hours,
        "attendance_rate": attendance_rate,
        "homework_score": homework_score,
        "sleep_hours": sleep_hours,
        "price": price,
    })

try:
    housing = fetch_california_housing(as_frame=True)
    df_raw = housing.frame.copy()
    df_raw = df_raw.rename(columns={"MedHouseVal": "price"})
    dataset_name = "California Housing Dataset"
except Exception as exc:
    print("在线数据下载失败，自动使用离线练习数据。错误信息:", exc)
    df_raw = make_offline_dataset()
    dataset_name = "Offline Student Score Dataset"

raw_path = DATA_DIR / "raw_dataset.csv"
df_raw.to_csv(raw_path, index=False)

print("数据集:", dataset_name)
print("原始数据已保存:", raw_path)
print("数据形状:", df_raw.shape)
df_raw.head()

## 3. 第一次查看数据

目标：知道数据有多少行、多少列、每列叫什么、预测目标是什么。

In [ ]:
df = pd.read_csv(DATA_DIR / "raw_dataset.csv")

print("数据形状:", df.shape)
print("字段名:", df.columns.tolist())
display(df.head())

In [ ]:
print("字段类型:")
display(df.dtypes)

print("缺失值数量:")
display(df.isnull().sum())

In [ ]:
df.describe()

## 4. EDA 可视化：特征分布

直方图可以帮助你观察每个字段的大致分布。

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns

axes = df[numeric_columns].hist(figsize=(12, 8), bins=30)
plt.suptitle("Feature Distributions")
plt.tight_layout()

feature_plot_path = FIGURE_DIR / "feature_distribution.png"
plt.savefig(feature_plot_path, dpi=150)
plt.show()

print("特征分布图已保存:", feature_plot_path)

## 5. 明确 Feature 和 Label

- Feature：输入给模型的字段。
- Label：模型要预测的答案。

本项目统一使用 `price` 作为 Label。

In [ ]:
LABEL_COLUMN = "price"
feature_columns = [column for column in df.columns if column != LABEL_COLUMN]

print("Feature columns:", feature_columns)
print("Label column:", LABEL_COLUMN)

## 6. 数据清洗

这里完成三件事：

1. 检查缺失值。
2. 用中位数填充数值缺失值。
3. 保存清洗后的数据。

In [ ]:
df_clean = df.copy()

for column in feature_columns + [LABEL_COLUMN]:
    if df_clean[column].isnull().any():
        median_value = df_clean[column].median()
        df_clean[column] = df_clean[column].fillna(median_value)
        print(f"字段 {column} 存在缺失值，已用中位数 {median_value:.4f} 填充")

clean_path = DATA_DIR / "clean_dataset.csv"
df_clean.to_csv(clean_path, index=False)

print("清洗后数据已保存:", clean_path)
print("剩余缺失值总数:", int(df_clean.isnull().sum().sum()))

## 7. 相关性分析

相关性热力图可以帮助你初步判断哪些字段可能和 `price` 更相关。

In [ ]:
corr = df_clean.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Heatmap")
plt.tight_layout()

heatmap_path = FIGURE_DIR / "correlation_heatmap.png"
plt.savefig(heatmap_path, dpi=150)
plt.show()

print("相关性热力图已保存:", heatmap_path)
display(corr[[LABEL_COLUMN]].sort_values(LABEL_COLUMN, ascending=False))

## 8. 准备建模数据

In [ ]:
X = df_clean[feature_columns]
y = df_clean[LABEL_COLUMN]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 9. 划分训练集和测试集

训练集用于让模型学习规律，测试集用于检查模型面对新数据时的效果。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("训练集:", X_train.shape, y_train.shape)
print("测试集:", X_test.shape, y_test.shape)

## 10. 训练线性回归模型

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("模型训练完成")
print("截距:", model.intercept_)

coef_df = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": model.coef_,
}).sort_values("coefficient", ascending=False)

coef_df

## 11. 模型预测

In [ ]:
y_pred = model.predict(X_test)

prediction_preview = pd.DataFrame({
    "ground_truth": y_test.values[:10],
    "prediction": y_pred[:10],
})

prediction_preview

## 12. 模型评估

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "value": [mae, rmse, r2],
})

metrics_path = RESULT_DIR / "metrics.csv"
metrics.to_csv(metrics_path, index=False)

print("评估指标已保存:", metrics_path)
metrics

指标理解：

- MAE：平均绝对误差，越小越好。
- RMSE：均方根误差，越小越好。
- R2：拟合优度，越接近 1 越好。

## 13. 可视化预测结果

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], color="red")

plt.xlabel("Ground Truth")
plt.ylabel("Prediction")
plt.title("Prediction vs Ground Truth")
plt.tight_layout()

prediction_plot_path = FIGURE_DIR / "prediction_vs_ground_truth.png"
plt.savefig(prediction_plot_path, dpi=150)
plt.show()

print("预测对比图已保存:", prediction_plot_path)

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Prediction")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.tight_layout()

residual_plot_path = FIGURE_DIR / "residual_plot.png"
plt.savefig(residual_plot_path, dpi=150)
plt.show()

print("残差图已保存:", residual_plot_path)

## 14. 查看误差最大的样本

这一步帮助你思考：模型在哪些样本上预测不好。

In [ ]:
error_df = X_test.copy()
error_df["ground_truth"] = y_test.values
error_df["prediction"] = y_pred
error_df["absolute_error"] = (error_df["ground_truth"] - error_df["prediction"]).abs()

error_df = error_df.sort_values("absolute_error", ascending=False)
error_df.head(10)

## 15. 保存模型并重新加载测试

保存模型后，后续项目或程序可以直接加载模型进行预测。

In [ ]:
model_path = MODEL_DIR / "linear_regression.pkl"
joblib.dump(model, model_path)

print("模型已保存:", model_path)
print("模型文件是否存在:", model_path.exists())

In [ ]:
loaded_model = joblib.load(model_path)

sample = X_test.iloc[[0]]
prediction = loaded_model.predict(sample)[0]
ground_truth = y_test.iloc[0]

print("单样本预测测试")
print("真实值:", ground_truth)
print("预测值:", prediction)

## 16. 最终检查

如果下面所有文件都存在，说明本周 Notebook 工程已经完整运行成功。

In [ ]:
expected_files = [
    DATA_DIR / "raw_dataset.csv",
    DATA_DIR / "clean_dataset.csv",
    FIGURE_DIR / "feature_distribution.png",
    FIGURE_DIR / "correlation_heatmap.png",
    FIGURE_DIR / "prediction_vs_ground_truth.png",
    FIGURE_DIR / "residual_plot.png",
    MODEL_DIR / "linear_regression.pkl",
    RESULT_DIR / "metrics.csv",
]

for path in expected_files:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status}: {path}")

## 17. 学习复盘

运行完成后，请尝试回答：

1. 什么是 Feature？
2. 什么是 Label？
3. 为什么要划分训练集和测试集？
4. 线性回归模型在训练时学到了什么？
5. MAE、RMSE、R2 分别表示什么？
6. Prediction vs Ground Truth 图怎么看？
7. Residual Plot 图怎么看？
8. 如果要改进模型，你会尝试什么？